In [1]:
import numpy as np
import cv2

import os
from os import path as osp
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms

from model.net import DGCNet

In [ ]:
IMG_PATH = '/home/boat/proxyISP/pytorch-superpoint/datasets/HPatches/wl_brownflora'
IMG_SIZE_DGC = (240, 240)

img1 = cv2.resize(cv2.cvtColor(cv2.imread(osp.join(IMG_PATH, '1.ppm'), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB),
                  IMG_SIZE_DGC)
img2 = cv2.resize(cv2.cvtColor(cv2.imread(osp.join(IMG_PATH, '2.ppm'), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB),
                  IMG_SIZE_DGC)

plt.figure()
ax1 = plt.subplot(1, 2, 1)
ax1.imshow(img1);
ax1.axis('off')
ax1.set_title('source image');

ax2 = plt.subplot(1, 2, 2)
ax2.imshow(img2);
ax2.axis('off')
ax2.set_title('target image (gt)');


In [ ]:
class DeNormalize:
    '''
    Removes normalization using the mean, std specified
    '''
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        for t, m, s in zip(tensor, self.mean, self.std):
            t.mul_(s).add_(m)
        return tensor


# Image pre-processing constants
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

# Image de-normalization
restore_image = DeNormalize(mean, std)

# Prepare data for the network
dataset_transforms = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

In [ ]:
use_cuda = torch.cuda.is_available()
device = torch.device('cuda:0' if use_cuda else 'cpu')

net = DGCNet()
net.load_state_dict(torch.load('/home/boat/proxyISP/DGC-Net/model/pretrained_models/dgc/checkpoint.pth',
                               map_location=torch.device(device))['state_dict'])
net.eval()
net.to(device);

In [ ]:
with torch.no_grad():
    # predict the warping grid between two images
    estimates_grid_pyr, _ = net(dataset_transforms(img1).unsqueeze(0).cuda(), dataset_transforms(img2).unsqueeze(0).cuda())
    # warp the source image based on the estimates to be aligned with the target view
    warp_img = F.grid_sample(dataset_transforms(img1).unsqueeze(0).cuda(), estimates_grid_pyr[-1].permute(0, 2, 3, 1).cuda())


# Visualize the results
plt.figure()
ax1 = plt.subplot(1, 3, 1)
ax1.imshow(img1);
ax1.axis('off')
ax1.set_title('source image');

ax2 = plt.subplot(1, 3, 2)
ax2.imshow(img2);
ax2.axis('off')
ax2.set_title('target image (gt)');

ax3 = plt.subplot(1, 3, 3)
ax3.imshow(restore_image(warp_img.squeeze()).permute(1, 2, 0).cpu().numpy());
ax3.axis('off')
ax3.set_title('target image (estimate)');


In [ ]:
with torch.no_grad():
    # predict the warping grid between two images
    estimates_grid_pyr, _ = net(dataset_transforms(img1).unsqueeze(0).cuda(), dataset_transforms(img2).unsqueeze(0).cuda())
    # warp the source image based on the estimates to be aligned with the target view
    warp_img = F.grid_sample(dataset_transforms(img1).unsqueeze(0).cuda(), estimates_grid_pyr[-1].permute(0, 2, 3, 1).cuda())


# Visualize the results
plt.figure()
ax1 = plt.subplot(1, 3, 1)
ax1.imshow(img1);
ax1.axis('off')
ax1.set_title('source image');

ax2 = plt.subplot(1, 3, 2)
ax2.imshow(img2);
ax2.axis('off')
ax2.set_title('target image (gt)');

ax3 = plt.subplot(1, 3, 3)
ax3.imshow(restore_image(warp_img.squeeze()).permute(1, 2, 0).cpu().numpy());
ax3.axis('off')
ax3.set_title('target image (estimate)');


# Compare 2 hpatches (original and optimized)

In [1]:
import numpy as np
import cv2

from model.net import DGCNet
def get_valid_pixel_mask(H, image_shape):
    h, w = image_shape[:2]

    Hinv = np.linalg.inv(H)

    y_grid, x_grid = np.indices((h, w))
    ones = np.ones_like(x_grid)
    coords = np.stack([x_grid, y_grid, ones], axis=-1).reshape(-1, 3).T  # Shape (3, N)

    mapped_coords = Hinv @ coords  # Shape (3, N)
    mapped_coords /= mapped_coords[2, :]  # Normalize by last row (homogeneous coords)

    x_mapped = mapped_coords[0, :]
    y_mapped = mapped_coords[1, :]

    valid = (
        (x_mapped >= 0) & (x_mapped < w) &
        (y_mapped >= 0) & (y_mapped < h)
    )

    # Convert to mask shape
    mask = valid.astype(np.uint8).reshape(h, w)
    return mask


In [ ]:
import os
import os.path as osp
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torchvision import transforms
from tqdm import tqdm
import random

# --------- Constants and Model Setup ---------
IMG_SIZE_DGC = (240, 240)

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

dataset_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

class DeNormalize:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        for t, m, s in zip(tensor, self.mean, self.std):
            t.mul_(s).add_(m)
        return tensor

restore_image = DeNormalize(mean, std)

def make_identity_grid(size, device):
    h, w = size
    grid_y, grid_x = torch.meshgrid(torch.linspace(-1, 1, h, device=device),
                                    torch.linspace(-1, 1, w, device=device), indexing='ij')
    grid = torch.stack((grid_x, grid_y), 2)
    return grid.permute(2, 0, 1).unsqueeze(0)  # shape: (1, 2, H, W)

# --------- Load Model ---------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

net = DGCNet()
checkpoint = torch.load('/home/boat/proxyISP/DGC-Net/model/pretrained_models/dgc/checkpoint.pth', map_location=device)
net.load_state_dict(checkpoint['state_dict'])
net.eval().to(device)

def scale_homography(H, scaley, scalex):
    if H.shape != (3, 3):
        raise ValueError("Input must be a 3x3 matrix.")

    S = np.array([
        [scalex, 0,     0],
        [0,     scaley, 0],
        [0,     0,     1]
    ], dtype=np.float32)

    S_inv = np.linalg.inv(S)
    return S @ H @ S_inv

def homography_str_to_numpy(h_str):
    lines = h_str.strip().split('\n')
    matrix = [list(map(float, line.strip().split())) for line in lines]
    return np.array(matrix, dtype=np.float32)

def load_image(path):
    original = cv2.imread(path)
    if original is None:
        raise FileNotFoundError(f"Image not found: {path}")
    ori_sz = original.shape
    img = cv2.resize(cv2.cvtColor(original, cv2.COLOR_BGR2RGB), IMG_SIZE_DGC)
    return img, ori_sz

def infer_and_warp(img1, img2):
    with torch.no_grad():
        tensor1 = dataset_transforms(img1).unsqueeze(0).to(device)
        tensor2 = dataset_transforms(img2).unsqueeze(0).to(device)
        flow_pyr, _ = net(tensor1, tensor2)
        grid = flow_pyr[-1].permute(0, 2, 3, 1)
        warped = F.grid_sample(tensor1, grid, align_corners=True)
        return restore_image(warped.squeeze()).clamp(0, 1).permute(1, 2, 0).cpu().numpy()

def alpha_blend_images(img1, img2, alpha=0.5):
    return cv2.addWeighted(img1, alpha, img2, 1 - alpha, 0)

def plot_comparison(seq_name, target_index, src1, tgt1, warp1, gt1, gt2, src2, tgt2, warp2, valid_mask, save_dir=None):
    plt.figure(figsize=(20, 10))
    plt.suptitle(f'Sequence: {seq_name} | Pair: 1 → {target_index}', fontsize=16)

    titles = ['Source A', 'Target A', 'Warped A', 'Ground Truth 1', 'Overlay A vs GT1',
              'Source B', 'Target B', 'Warped B', 'Ground Truth 2', 'Overlay B vs GT2']

    warp1 = np.round((warp1 * 255)).astype(np.uint8)
    warp2 = np.round((warp2 * 255)).astype(np.uint8)

    if valid_mask is not None:
        valid_mask = valid_mask == 1
        warp1[~valid_mask] = 0
        warp2[~valid_mask] = 0
        gt1[~valid_mask] = 0
        gt2[~valid_mask] = 0

    # Convert GT and warped to pure channels for overlay
    overlay_a = np.zeros_like(gt1)
    overlay_a[..., 0] = warp1[..., 0]  # Blue channel from warped
    overlay_a[..., 2] = gt1[..., 0]    # Red channel from GT

    overlay_b = np.zeros_like(gt2)
    overlay_b[..., 0] = warp2[..., 0]  # Blue from warped
    overlay_b[..., 2] = gt2[..., 0]    # Red from GT

    images = [src1, tgt1, warp1, gt1, overlay_a,
              src2, tgt2, warp2, gt2, overlay_b]

    for i, img in enumerate(images):
        plt.subplot(2, 5, i + 1)
        plt.imshow(img)
        plt.axis('off')
        plt.title(titles[i])

    plt.tight_layout(rect=[0, 0, 1, 0.95])

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        save_path = osp.join(save_dir, f"{seq_name}_1to{target_index}_overlay.png")
        plt.savefig(save_path)
        print(f"Saved plot to {save_path}")

    plt.show()

def visualize_matching_sequences_multi(
    hpatches_roots, num_sequences=5, shuffle=True,
    allowed_view_indices=None, save_dir=None, seq_names = None
):
    """
    hpatches_roots: list of dataset root paths
    """
    if len(hpatches_roots) < 2:
        print("Need at least 2 HPatches roots for comparison.")
        return

    # Find common sequence names across all roots
    seq_names_sets = [set(os.listdir(root)) for root in hpatches_roots]
    matched_seqs = sorted(list(set.intersection(*seq_names_sets)))

    if seq_names is not None:
        matched_seqs = seq_names

    if not matched_seqs:
        print("No matching sequences found.")
        return

    print(f"Found {len(matched_seqs)} matching sequences. Showing up to {num_sequences}.")

    if shuffle:
        random.shuffle(matched_seqs)

    if allowed_view_indices is None:
        allowed_view_indices = [2, 3, 4, 5, 6]

    for seq_name in tqdm(matched_seqs[:num_sequences]):
        imgs_original = []
        imgs_resized = []
        gt_imgs = []

        # Load the "1.ppm" source image for each dataset
        for root in hpatches_roots:
            path = osp.join(root, seq_name)
            try:
                img, original_size = load_image(osp.join(path, '1.ppm'))
            except FileNotFoundError as e:
                print(f"Skipping sequence {seq_name}: {e}")
                continue
            imgs_original.append(img)
            imgs_resized.append(img)  # placeholder; could resize differently per dataset

        for i in range(2, 7):
            if i not in allowed_view_indices:
                continue

            tgt_imgs = []
            valid_masks = []
            warped_imgs = []
            gt_imgs = []

            for idx, root in enumerate(hpatches_roots):
                path = osp.join(root, seq_name)
                tgt_path = osp.join(path, f"{i}.ppm")
                if not osp.exists(tgt_path):
                    print(f"Missing {i}.ppm in {seq_name} for dataset {idx}, skipping.")
                    continue

                tgt_img, original_size = load_image(tgt_path)
                tgt_imgs.append(tgt_img)

                # Load homography
                try:
                    with open(osp.join(path, f"H_1_{i}")) as f:
                        H = homography_str_to_numpy(f.read())
                except Exception as e:
                    print(f"Failed to read homography H_1_{i} in {seq_name} for dataset {idx}: {e}")
                    continue

                # Scale homography to resized images
                H = scale_homography(H, IMG_SIZE_DGC[0]/original_size[0], IMG_SIZE_DGC[1]/original_size[1])
                valid_mask = get_valid_pixel_mask(H, tgt_img.shape)
                valid_masks.append(valid_mask)

                # Warp source image using DGC-Net
                warped = infer_and_warp(imgs_original[idx], tgt_img)
                warped_imgs.append(warped)

                # Warp using GT homography
                gt = cv2.warpPerspective(imgs_original[idx], H, (IMG_SIZE_DGC[1], IMG_SIZE_DGC[0]))
                gt_imgs.append(gt)

            # Plot comparison
            plt.figure(figsize=(20, 5 * len(hpatches_roots)))
            plt.suptitle(f'Sequence: {seq_name} | Pair: 1 → {i}', fontsize=16)
            
            for row, (root_name, src, tgt, warp, gt, mask) in enumerate(zip(
                    [osp.basename(r) for r in hpatches_roots],
                    imgs_original, tgt_imgs, warped_imgs, gt_imgs, valid_masks)):
            
                if mask is not None:
                    mask = mask == 1
                    warp[~mask] = 0
                    gt[~mask] = 0
            
                overlay = np.zeros_like(gt)
                overlay[..., 0] = warp[..., 0] * 255.0
                overlay[..., 2] = gt[..., 0]
            
                images = [src, tgt, warp, gt, overlay]
                titles = ['Source', 'Target', 'Warped', 'Ground Truth', 'Overlay']
            
                for col, img in enumerate(images):
                    ax = plt.subplot(len(hpatches_roots), 5, row * 5 + col + 1)
                    plt.imshow(img)
                    plt.axis('off')
            
                    # # Only add column titles on first row
                    # if row == 0:
                    #     plt.title(titles[col], fontsize=12)
            
                    # # Add dataset name below each row (on first column)
                    # if col == 0:
                    #     plt.text(0.5, -0.15, root_name, fontsize=12,
                    #              rotation=0, transform=ax.transAxes)
            
            plt.tight_layout(rect=[0, 0, 1, 0.95])
            if save_dir:
                os.makedirs(save_dir, exist_ok=True)
                save_path = osp.join(save_dir, f"{seq_name}_1to{i}_overlay.png")
                plt.savefig(save_path)
                print(f"Saved plot to {save_path}")
            plt.show()



# === PARAMETERS ===
# HP_ROOTS = [i.path for i in os.scandir("/home/boat/proxyISP/pytorch-superpoint/datasets/HPatches_caches_DGC")]
# hp_root3 = "/home/boat/proxyISP/pytorch-superpoint/datasets/HPatches_caches_DGC/eval_ll_FIXZEROGRADBUG_CFANORMALIZE_train_v16.2-chroma-HumanTunedInitialHype_lowlight_pooled480x640_allHomoRepeatedRaw_standardize_lr0.0005_gradac32_123000_HpatchesV4"
# hp_root2 = "/home/boat/proxyISP/pytorch-superpoint/datasets/HPatches_caches_DGC/eval_ll_v16.2-chroma-HumanTunedInitialHype_replicate-s21fe_lowlight_lr0.0005_schedulerPlateauTo0.00001_bs1_ga8_45000_HpatchesV4"
# hp_root1 = "/home/boat/proxyISP/pytorch-superpoint/datasets/HPatches_caches_DGC/eval_ll_v16.2-chroma-ISPDefaultInitialHype_original_HpatchesV4"
hp_root3 = "/home/boat/proxyISP/pytorch-superpoint/datasets/HPatches_caches_DGC/eval_sl_FIXZEROGRADBUG_CFANORMALIZE_train_v16.2-chroma-HumanTunedInitialHype_sunlit_pooled480x640_allHomoRepeatedRaw_standardize_lr0.0005_gradac32_105000_HpatchesV4"
hp_root2 = "/home/boat/proxyISP/pytorch-superpoint/datasets/HPatches_caches_DGC/eval_sl_v16.2-chroma-HumanTunedInitialHype_replicate-s21fe_sunlit_lr0.0005_schedulerPlateauTo0.00001_bs1_ga8_120000_HpatchesV4"
hp_root1 = "/home/boat/proxyISP/pytorch-superpoint/datasets/HPatches_caches_DGC/eval_sl_v16.2-chroma-ISPDefaultInitialHype_original_HpatchesV4"

HP_ROOTS = [hp_root3,hp_root2,hp_root1]
# Call with filter (example: only viewpoints 2 and 5)
visualize_matching_sequences_multi(
    HP_ROOTS,
    num_sequences=20,
    allowed_view_indices=[5],
    seq_names = ["sl_qrcode"],
    save_dir='/home/boat/proxyISP/DGC-Net/visualize_sl_HPatchesV4'
)



/tmp/ipykernel_843245/4266253537.py:46: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('/home/boat/proxyISP/DGC-Net/model/pretrained_models/dgc/checkp

Found 1 matching sequences. Showing up to 20.


  0%|                                                                                                      | 0/1 [00:00<?, ?it/s]

Saved plot to /home/boat/proxyISP/DGC-Net/visualize_sl_HPatchesV4/sl_qrcode_1to5_overlay.png
